# JEPA on recurrence plots: RP versus attention

## 1. Setup, hypothesis, and pre-registered decision rule

This study tests whether a recurrence plot makes associative recall available to a local convolution. The main comparison is recurrence representation plus local mixing versus raw-series attention and raw-series convolution. Runs are cached so an interrupted full benchmark can resume.

H2 is supported if, on recall with N=64, four key–value pairs, width d=32, depth 2, and the mean of seeds 0, 1, and 2, the RP-CNN using D+series has R² at least the raw-attention R² minus 0.05, and both models exceed raw convolution by at least 0.30 R². The secondary rule applies the same test to RP-S4ND, replacing raw convolution with raw Hyena. We report forward FLOPs per input window as well as quality; RP encoders pay a quadratic-in-N input-representation cost. This rule is fixed before inspecting results.


In [ ]:
from pathlib import Path
import sys, json
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from jepa_experiments.data import TASKS
from rpjepa.encoders import ENCODERS, make_encoder, count_flops
from jepa_experiments.train import train_supervised
from jepa_experiments.experiment import SMOKE, get_split, cached_run

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
M, TAU, DEPTH, BATCH, LR = 3, 1, 2, 128, 1e-3
FULL_SIZES = {'train': 20000, 'val': 2000, 'test': 5000}
FULL_SEEDS = (0, 1, 2)
FULL_EPOCHS, FULL_PATIENCE = 40, 10
SIZES = {'train': 800, 'val': 200, 'test': 300} if SMOKE else FULL_SIZES
SEEDS = (0,) if SMOKE else FULL_SEEDS
EPOCHS, PATIENCE = (2, 1) if SMOKE else (FULL_EPOCHS, FULL_PATIENCE)
CACHE_ROOT = Path('cache/step2')
TABLE_ROOT = Path('tables/step2') / ('smoke' if SMOKE else '')
CACHE_ROOT.mkdir(parents=True, exist_ok=True); TABLE_ROOT.mkdir(parents=True, exist_ok=True)
D_RUN = 16 if SMOKE else 32
SMOKE_SPECS = ('raw_attn', 'raw_conv', 'rp_cnn', 'recattn')
print({'device': DEVICE, 'smoke': SMOKE, 'sizes': SIZES, 'seeds': SEEDS, 'epochs': EPOCHS, 'patience': PATIENCE})

def get_data(task, split, n_states, seed, n_pairs):
    difficulty = {'n_pairs': int(n_pairs)} if task == 'recall' else {}
    return get_split(task, split, SIZES[split], n_states, M, TAU, base_seed=seed, **difficulty)

def run_one(spec_label, enc_name, enc_kw, task, n_states, d, n_pairs, seed, epochs=EPOCHS, patience=PATIENCE):
    tr, va, te = (get_data(task, split, n_states, seed, n_pairs) for split in ('train', 'val', 'test'))
    key = {'spec_label': spec_label, 'enc_name': enc_name, 'enc_kw': enc_kw, 'd': int(d), 'depth': DEPTH,
           'N': int(n_states), 'n_pairs': int(n_pairs), 'task': task, 'seed': int(seed), 'epochs': int(epochs),
           'patience': int(patience), 'sizes': SIZES, 'batch': BATCH, 'lr': LR, 'm': M, 'tau': TAU}
    result = cached_run(CACHE_ROOT, key, lambda: train_supervised(
        enc_name, d, DEPTH, tr, va, te, n_states=n_states, m=M, tau=TAU, epochs=epochs, batch=BATCH,
        lr=LR, seed=seed, patience=patience, device=DEVICE, enc_kw=enc_kw))
    return {'spec': spec_label, 'encoder': enc_name, 'enc_kw': json.dumps(enc_kw, sort_keys=True),
            'task': task, 'N': n_states, 'd': d, 'depth': DEPTH, 'n_pairs': n_pairs, 'seed': seed,
            'epochs': epochs, 'patience': patience, **result}

def run_configs(configs):
    rows = []
    for i, c in enumerate(configs, 1):
        row = run_one(**c)
        rows.append(row)
        print(f"{i}/{len(configs)} {row['task']} {row['spec']} N={row['N']} d={row['d']} pairs={row['n_pairs']} seed={row['seed']} train_s={row['train_s']:.2f}")
    return pd.DataFrame(rows)

def cfg(spec, task, n, d, pairs, seed, epochs=EPOCHS, patience=PATIENCE):
    enc_name, enc_kw = SPECS[spec]
    return dict(spec_label=spec, enc_name=enc_name, enc_kw=enc_kw, task=task, n_states=n, d=d,
                n_pairs=pairs, seed=seed, epochs=epochs, patience=patience)


## 2. Encoder specifications

The raw models see delay-embedded time-series states. RP models receive the corresponding two-dimensional representation and return row-pooled time tokens. Exact recurrence attention is the default; its Taylor approximation is listed separately.


In [ ]:
SPECS = {
    'raw_attn': ('raw_attn', {}),
    'raw_conv': ('raw_conv', {}),
    'raw_hyena': ('raw_hyena', {}),
    'rp_cnn': ('rp_cnn', {'rp_kind': 'D+series'}),
    'rp_cnn_D': ('rp_cnn', {'rp_kind': 'D'}),
    'rp_cnn_RPstack': ('rp_cnn', {'rp_kind': 'RPstack+series'}),
    'rp_s4nd': ('rp_s4nd', {'rp_kind': 'D+series'}),
    'rp_vit': ('rp_vit', {'rp_kind': 'D+series', 'pos': 'grid'}),
    'recattn': ('recattn', {'exact': True}),
    'recattn_taylor': ('recattn_taylor', {}),
}
assert all(name in ENCODERS for name, _ in SPECS.values())
print(SPECS)


## 3. Width sweep

Recall uses four key–value pairs. Widths and model families follow the registered grid; RP-CNN, RP-S4ND, and RP-ViT stop at width 64 because their full-resolution two-dimensional input is expensive. The first plot compares R² with width, and the second compares R² with forward FLOPs per window. RP-S4ND at width 64 runs one seed only (about 23 min per run). The table gives the smallest width reaching R² ≥ 0.9, or ‘not reached’.


In [ ]:
raw_width_specs = ('raw_attn', 'raw_conv', 'raw_hyena', 'recattn')
rp_width_specs = ('rp_cnn', 'rp_s4nd', 'rp_vit')
width_configs = []
width_n = 32 if SMOKE else 64
widths_raw = (8, 16) if SMOKE else (8, 16, 32, 64, 128, 256)
widths_rp = (8, 16) if SMOKE else (8, 16, 32, 64)
width_specs = tuple(s for s in raw_width_specs + rp_width_specs if not SMOKE or s in SMOKE_SPECS)
for spec in width_specs:
    for d in (widths_rp if spec in rp_width_specs else widths_raw):
        seeds = SEEDS[:1] if (spec, d) == ('rp_s4nd', 64) else SEEDS
        for seed in seeds: width_configs.append(cfg(spec, 'recall', width_n, d, 4, seed))
width_df = run_configs(width_configs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for spec, group in width_df.groupby('spec'):
    means = group.groupby('d').metric.mean().sort_index()
    axes[0].plot(means.index, means.values, marker='o', label=spec)
    by_flops = group.groupby('flops').metric.mean().sort_index()
    axes[1].plot(by_flops.index, by_flops.values, marker='o', label=spec)
axes[0].set(xscale='log', xlabel='Width d', ylabel='Test R²', title='Quality versus width')
axes[1].set(xscale='log', xlabel='Forward FLOPs per window', ylabel='Test R²', title='Quality versus compute')
for ax in axes: ax.grid(True, alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
width_threshold_rows = []
for spec, group in width_df.groupby('spec'):
    means = group.groupby('d').agg(metric=('metric','mean'), flops=('flops','mean')).sort_index()
    reached = means[means.metric >= .9]
    width_threshold_rows.append({'spec': spec, 'smallest_d': int(reached.index[0]) if len(reached) else 'not reached',
                                'flops_per_window': float(reached.iloc[0].flops) if len(reached) else np.nan})
width_threshold_df = pd.DataFrame(width_threshold_rows)
display(width_threshold_df)


## 4. Sequence-length sweep

At fixed width 32 and four pairs, measure how quality and per-window FLOPs change with N. RP-S4ND is skipped at N=128: its 2D FFT path measured 20 s per training step there (about 35 h per run). This sweep also excludes the representation-specific ablations and Taylor recurrence attention listed in the protocol.


In [ ]:
length_specs_full = tuple(s for s in SPECS if s not in ('rp_cnn_D', 'rp_cnn_RPstack', 'recattn_taylor'))
length_specs = tuple(s for s in length_specs_full if not SMOKE or s in SMOKE_SPECS)
length_values = (32,) if SMOKE else (32, 64, 128)
length_configs = [cfg(s, 'recall', n, D_RUN, 4, seed) for n in length_values for s in length_specs for seed in SEEDS
                  if (s, n) != ('rp_s4nd', 128)]
length_df = run_configs(length_configs)
display(length_df[['spec','N','seed','metric','flops','train_s']])


## 5. Number-of-pairs sweep

At N=64 and width 32, vary the number of planted key–value pairs. RP-S4ND runs one seed except at four pairs (shared with the width sweep).


In [ ]:
pair_values = (1, 4) if SMOKE else (1, 2, 4, 8)
pairs_n = 32 if SMOKE else 64
pairs_specs = tuple(s for s in length_specs_full if not SMOKE or s in SMOKE_SPECS)
pairs_configs = [cfg(s, 'recall', pairs_n, D_RUN, pairs, seed) for pairs in pair_values for s in pairs_specs
                 for seed in (SEEDS[:1] if s == 'rp_s4nd' and pairs != 4 else SEEDS)]
pairs_df = run_configs(pairs_configs)
display(pairs_df[['spec','n_pairs','seed','metric','flops','train_s']])


## 6. Input-representation ablation

Compare distance-only RP input, distance plus the original series channels, thresholded multi-recurrence plots plus series, and raw attention. This tests whether the series channels help and whether thresholding discards useful information. The task set includes recall and the other benchmark tasks.


In [ ]:
ablation_specs_full = ('rp_cnn', 'rp_cnn_D', 'rp_cnn_RPstack', 'raw_attn')
ablation_specs = tuple(s for s in ablation_specs_full if not SMOKE or s in SMOKE_SPECS)
ablation_tasks = tuple(TASKS)
ablation_n = 32 if SMOKE else 64
ablation_seeds = (0,) if SMOKE else (0, 1)
ablation_configs = [cfg(s, task, ablation_n, D_RUN, 4, seed) for task in ablation_tasks for s in ablation_specs for seed in ablation_seeds]
ablation_df = run_configs(ablation_configs)
display(ablation_df[['task','spec','seed','metric','flops','train_s']])


## 7. Beyond recall

Test system identification, Lyapunov-exponent prediction, forecasting, and changepoint localization at width 16. Width 64 results for the same encoders are in Step 1 (`step1_supervised.csv`, same data sizes); the distance-only and thresholded RP-CNN variants are covered at width 32 in section 6. This section uses 30 epochs with patience 5.


In [ ]:
other_tasks = ('system', 'lyapunov', 'forecast', 'changepoint')
other_specs_full = tuple(s for s in SPECS if s != 'recattn_taylor')
other_specs = tuple(s for s in other_specs_full if not SMOKE or s in SMOKE_SPECS)
other_widths = (8, 16) if SMOKE else (16,)
other_n = 32 if SMOKE else 64
other_epochs, other_patience = (2, 1) if SMOKE else (30, 5)
other_seeds = (0,) if SMOKE else (0, 1)
other_configs = [cfg(s, task, other_n, d, 4, seed, other_epochs, other_patience)
                 for task in other_tasks for d in other_widths for s in other_specs for seed in other_seeds]
other_df = run_configs(other_configs)
display(other_df[['task','spec','d','seed','metric','flops','train_s']])


## 8. Pre-registered decision

Evaluate the rule from section 1 using the width-sweep recall results at N=64 and d=32. In smoke mode the requested N, width, and three seeds are deliberately reduced; the printed smoke status is only a preview and is not the registered full-run decision.


In [ ]:
def decision_value(df, spec):
    rows = df[(df.spec == spec) & (df.task == 'recall') & (df.d == (16 if SMOKE else 32)) & (df.N == (32 if SMOKE else 64))]
    return (float(rows.metric.mean()), float(rows.metric.std(ddof=1)) if len(rows) > 1 else np.nan, int(len(rows))) if len(rows) else (np.nan, np.nan, 0)

comparisons = [('primary', 'rp_cnn', 'raw_attn', 'raw_conv'), ('secondary', 'rp_s4nd', 'raw_attn', 'raw_hyena')]
decision_rows = []
for label, rp_spec, attention_spec, local_spec in comparisons:
    rp_mean, rp_sd, rp_n = decision_value(width_df, rp_spec)
    attn_mean, attn_sd, attn_n = decision_value(width_df, attention_spec)
    local_mean, local_sd, local_n = decision_value(width_df, local_spec)
    evaluable = min(rp_n, attn_n, local_n) > 0
    supported = bool(evaluable and rp_mean >= attn_mean - .05 and rp_mean >= local_mean + .30 and attn_mean >= local_mean + .30)
    status = ('SUPPORTED' if supported else 'NOT SUPPORTED') if evaluable else 'NOT SUPPORTED (not evaluable in smoke)'
    decision_rows.append({'comparison': label, 'RP_spec': rp_spec, 'attention_spec': attention_spec, 'local_spec': local_spec,
        'RP_mean_R2': rp_mean, 'RP_sd': rp_sd, 'attention_mean_R2': attn_mean, 'attention_sd': attn_sd,
        'local_mean_R2': local_mean, 'local_sd': local_sd, 'seeds_per_model_min': min(rp_n, attn_n, local_n),
        'status': status, 'rp_flops_per_window': float(width_df.loc[(width_df.spec == rp_spec) & (width_df.d == (16 if SMOKE else 32)) & (width_df.N == (32 if SMOKE else 64)), 'flops'].mean()) if rp_n else np.nan})
decision_df = pd.DataFrame(decision_rows)
display(decision_df)
def metric_summary(mean, sd, count):
    if not count: return 'not measured'
    return f'{mean:.3f} ± {sd:.3f}' if np.isfinite(sd) else f'{mean:.3f} ± unavailable (n=1)'
for row in decision_rows:
    print(f"{row['comparison'].upper()}: {row['status']} | RP {metric_summary(row['RP_mean_R2'], row['RP_sd'], row['seeds_per_model_min'])}; attention {metric_summary(row['attention_mean_R2'], row['attention_sd'], row['seeds_per_model_min'])}; local {metric_summary(row['local_mean_R2'], row['local_sd'], row['seeds_per_model_min'])}")


## 9. Tables and run times

Write the five result tables and the decision table, and show the measured training time per encoder, N and width. Each cache identity includes the encoder specification, architecture and data settings, seed, training schedule and split sizes. Smoke results and caches are kept in `smoke/` subfolders.


In [ ]:
width_df.to_csv(TABLE_ROOT/'width_sweep.csv', index=False)
length_df.to_csv(TABLE_ROOT/'length_sweep.csv', index=False)
pairs_df.to_csv(TABLE_ROOT/'pairs_sweep.csv', index=False)
ablation_df.to_csv(TABLE_ROOT/'representation_ablation.csv', index=False)
other_df.to_csv(TABLE_ROOT/'other_tasks.csv', index=False)
decision_df.to_csv(TABLE_ROOT/'decision.csv', index=False)

all_runs = pd.concat([width_df, length_df, pairs_df, ablation_df, other_df], ignore_index=True)
display(all_runs.groupby(['spec', 'N', 'd']).train_s.agg(runs='count', mean_s='mean').round(1).reset_index())
print('Tables:', TABLE_ROOT.resolve())


## 10. Findings

Full run: 106 width-sweep, 60 length, 78 pairs, 48 ablation and 72 other-task runs; recall uses 20,000/2,000/5,000 windows, 40 epochs, patience 10. Figures are in `results/tables/jepa/step2/figures/`. R² on test, mean (SD) over seeds.

**Pre-registered decision: H2 supported (primary and secondary).** At recall, N = 64, four pairs, d = 32:

| | R² (3 seeds) | FLOPs / window |
|---|---|---|
| rp_cnn (D + series, 3×3 CNN) | 0.998 (0.000) | 287 M |
| rp_s4nd | 0.840 (0.247) | 188 M |
| raw_attn | 0.763 (0.075) | 2.1 M |
| raw_conv | 0.006 (0.005) | 2.4 M |
| raw_hyena | 0.004 (0.005) | 2.9 M |

The secondary rule passes on the mean, but RP-S4ND is unstable: one seed reached 0.55 and the other two ≈ 0.98.

1. **A precomputed RP makes recall easy for a local model, at every scale tested.** rp_cnn stays at R² ≥ 0.994 for every width (d = 8 to 64), every length (N = 32 to 128) and every number of pairs (1 to 8). Raw attention gets worse as the task gets harder: 0.909 → 0.529 from N = 32 to 128, and 0.924 → 0.521 from 1 to 8 pairs. Local convolution and gated long convolution (Hyena) on the raw series never learn recall (R² ≤ 0.1). This is the Zoology result, and the RP removes it.
2. **Cost is the catch.** Per window, rp_cnn costs about 140× the FLOPs of raw attention at the same width, and the cost grows with N² (72 M → 1,148 M FLOPs from N = 32 to 128). Counted by quality per FLOP, however, the smallest model reaching R² ≥ 0.9 is about as cheap for rp_cnn (d = 8: 18 M FLOPs, 0.994) as for attention (d = 128: 34 M FLOPs, 0.936) or exact recurrence attention (d = 64: 17 M FLOPs, 0.967).
3. **Which part of the RP does the work?** (section 6, d = 32)
   - **Distances alone cannot do recall.** rp_cnn on D only scores R² 0.000. D shows which earlier state matches the query, but not the value that followed it. Forecast also drops from 0.982 to 0.485. The series channels are essential: an RP describes relative geometry, not levels.
   - **Thresholding costs little.** Binary RPs at four recurrence rates plus the series reach 0.967 on recall and match unthresholded D on the other tasks.
4. **Not every 2D encoder can use the RP.** rp_vit (8×8 patches) fails recall at every width (≤ 0.10). Patching blurs the single matching cell in the query row. rp_s4nd needs d ≥ 32, fails at 8 pairs (0.003), and was not run at N = 128 (20 s per training step).
5. **Exact recurrence attention ≈ attention, plus a kernel that matches by distance.** It beats raw attention at d = 32–64 (0.884 and 0.967 vs 0.763 and 0.860). But it fails at N = 128 (0.012) and at d ≥ 128 on some seeds. Raw attention also fails at d = 256 (0.017). [INFERENCE] These failures look like optimisation instability at the fixed learning rate of 1e-3, not a lack of capacity: some seeds at d = 128 reach 0.98–0.99 while others stay near 0.
6. **Beyond recall, the differences are small.** At d = 16, all encoders are within about 0.03 of each other on system and lyapunov. rp_cnn is best on forecast (0.963 vs 0.941 for attention), and exact recattn is best on lyapunov (0.936) and system (0.854). changepoint stays hard for everyone (≤ 0.16).

**Caveats.**
- One learning rate and one depth.
- The width-256 and N = 128 failures of the attention models were not tuned away.
- Some configurations ran fewer seeds: rp_s4nd at d = 64 ran one seed, and so did rp_s4nd outside four pairs.
